# Task 1 -- Beam Search vs Greedy Decoding (zero-shot)

Small, throwaway experiment notebook (branch `beamSearchTry`): runs the **zero-shot** Task 1 baseline on a **small subsample** of the test set, comparing three decoding settings:

1. **Greedy** (`Task_1.ipynb`'s current default) -- deterministic, picks the single best next token every step, can't recover from an early mistake.
2. **Beam search** (`num_beams=4`) -- keeps 4 candidate sequences at each step, should find higher joint-probability outputs than greedy.
3. **Beam search + `no_repeat_ngram_size`** -- beam search alone doesn't reliably stop the model's `ppppppp...` repetition loops (all beams tend to converge on the same loop); this constraint forbids repeating a 3-token n-gram, to check whether it's the missing piece.

Only the **zero-shot vanilla model** is evaluated here -- no fine-tuning, no dataset writes, no Hub pushes. Point is to see whether decoding strategy alone moves the needle before spending GPU time changing it in the real training notebooks.

Downloading Libraries and Imports

In [ ]:
!rm -rf /usr/local/lib/python3.13/dist-packages/~orch*
!pip install -q --upgrade "pillow<11.0.0" torch torchvision transformers>=4.45.0 accelerate torchao scikit-learn tqdm chess cairosvg python-Levenshtein datasets peft huggingface_hub python-dotenv


!pip uninstall -y torchaudio -q


!pip install -q flash-linear-attention

# Standard library imports
import json
import os
import subprocess
import sys
from pathlib import Path

# Third-party library imports
import pandas as pd
import torch
from datasets import load_dataset
from huggingface_hub import login
from tqdm import tqdm
from transformers import AutoModelForImageTextToText, AutoProcessor


Setting up environment

In [ ]:
# Project configuration dictionary
CONFIG = {
    "colab": True,
    "branch": "beamSearchTry",
    "repo_name": "BigDataAndTextMiningProject",
    "repo_owner": "Aivon99",
    "repo_dir": "/content/BigDataAndTextMiningProject",
}

# 1. Setup repository path and handle cloning safely
repo_root = Path(CONFIG["repo_dir"])

if CONFIG["colab"]:
    if repo_root.exists():
        # /content survives "Restart session", so an old clone can linger -- pull so
        # the notebook always runs the latest code (restart the session afterwards if
        # eval.utilities was already imported in this kernel).
        print(f"Repository directory already exists at: {repo_root}; pulling latest changes...")
        pull = subprocess.run(["git", "-C", str(repo_root), "pull", "--ff-only"], capture_output=True, text=True)
        print((pull.stdout or pull.stderr).strip())
    else:
        auth_url = "https://"
        repo_url = f"{auth_url}github.com/{CONFIG['repo_owner']}/{CONFIG['repo_name']}.git"

        print(f"Cloning repository from {repo_url}...")
        result = subprocess.run(
            ["git", "clone", "--single-branch", "--branch", CONFIG["branch"], repo_url, str(repo_root)],
            capture_output=True, text=True
        )
        assert result.returncode == 0, f"Git clone failed: {result.stderr}"
else:
    repo_root = Path(".").resolve().parent.parent

print(f"Setup Complete. REPO_ROOT: {repo_root}")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "src"))
sys.path.insert(0, str(repo_root / "src" / "data"))
sys.path.insert(0, str(repo_root / "src" / "eval"))

from eval.utilities import evaluate_chessboard_model_task_1

print("All custom modules imported successfully!")


Authenticate and load a small subsample of the Task 1 test set

In [ ]:
hf_token = None
if CONFIG["colab"]:
    try:
        from google.colab import userdata
        hf_token = userdata.get("HF_TOKEN")
    except Exception:
        hf_token = None
else:
    from dotenv import load_dotenv
    load_dotenv(repo_root / ".env")
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Logged in to Hugging Face Hub using HF_TOKEN.")
else:
    print("No HF_TOKEN found in secrets/.env/environment -- falling back to interactive login.")
    login()

# Small subsample -- this notebook is about comparing decoding strategies, not
# producing a publishable benchmark number, so a fraction of the full test set
# (400 samples) is enough and keeps each experiment to a couple of minutes.
NUM_SAMPLES = 40

dataset_name = "bdatm-project/dataset_task1"
print(f"Downloading dataset '{dataset_name}' (test split only)...")
test_split_full = load_dataset(dataset_name, split="test")
test_split = test_split_full.select(range(min(NUM_SAMPLES, len(test_split_full))))

print(f"\nUsing {len(test_split)} / {len(test_split_full)} test samples.")
print(test_split[0])


Load the vanilla model

In [ ]:
model_id = "Qwen/Qwen3.5-0.8B"
print(f"Loading model {model_id}...")

model_vanilla = AutoModelForImageTextToText.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

processor = AutoProcessor.from_pretrained(model_id)
print("Model and processor loaded correctly!")


## Experiment 1: Greedy decoding (baseline)

Reproduces `Task_1.ipynb`'s current zero-shot behaviour: `generate_kwargs={}` means `evaluate_chessboard_model_task_1` falls back to its default (`max_new_tokens=100`, everything else at the library default of `do_sample=False`, i.e. greedy).

In [ ]:
greedy_results_df, greedy_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Greedy",
)

all_results = greedy_summary_df
display(greedy_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


## Experiment 2: Beam search

`num_beams=4` keeps 4 candidate sequences at every step instead of committing to the single best token; `early_stopping=True` stops once all beams have produced an EOS token rather than always running to `max_new_tokens`.

In [ ]:
beam_results_df, beam_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Beam Search (n=4)",
    generate_kwargs={"num_beams": 4, "early_stopping": True},
)

all_results = pd.concat([all_results, beam_summary_df], ignore_index=True)
display(beam_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


## Experiment 3: Beam search + `no_repeat_ngram_size`

Beam search alone doesn't reliably break the `ppppppp...` repetition loops seen in the greedy baseline, since every beam tends to converge on the same locally-attractive loop. `no_repeat_ngram_size=3` forbids repeating any 3-token sequence, which directly targets that failure mode -- this checks whether it's the missing piece rather than beam width itself.

In [ ]:
beam_norepeat_results_df, beam_norepeat_summary_df = evaluate_chessboard_model_task_1(
    model=model_vanilla,
    processor=processor,
    dataset_split=test_split,
    model_name="Zero-Shot Beam Search (n=4) + no-repeat-3gram",
    generate_kwargs={"num_beams": 4, "early_stopping": True, "no_repeat_ngram_size": 3},
)

all_results = pd.concat([all_results, beam_norepeat_summary_df], ignore_index=True)
display(beam_norepeat_results_df[["sample_id", "ground_truth", "predicted", "fen_exact_match", "square_by_square_accuracy"]].head())
display(all_results)


## Comparison

`fen_exact_match` is expected to stay near 0 for a zero-shot 0.8B model regardless of decoding strategy (it requires every character of the FEN to match). `square_by_square_accuracy` and `character_error_rate` are the more informative columns here -- do they move at all between greedy and beam search on this subsample?

In [ ]:
sorted_results = all_results.sort_values(by="character_error_rate", ascending=True).reset_index(drop=True)
print(f"--- Decoding Strategy Comparison (Zero-Shot, n={len(test_split)} samples) ---")
display(sorted_results)
